# Mass flash erase with password on MCXC353

The MCXC353 device supports a password-protected flash mass-erase that can be invoked via the
Debug Mailbox even after the device has been moved to an in-field lifecycle state (IN_FIELD1 /
IN_FIELD2). The flow consists of programming the CMPA password region with a 128-bit
``ERASE_PASSWORD`` token, transitioning the device to an in-field lifecycle state, and finally
issuing ``nxpdebugmbox cmd erase --password`` over a debug probe. The ROM validates the four
32-bit words sent through the mailbox against the value stored in CMPA and erases the flash only
when both match and the password-region header marker (``0x5057``) is present.

MCXC353 does **not** support ISP mode (`isp.rom.protocol: null` in the device database), so
``blhost``/``pfr write``/``pfr read`` cannot be used at all for this device - there is no UART/USB
ISP entry point to connect to. All CMPA provisioning and lifecycle inspection must go through the
debug (SWD) interface instead:

- CMPA binaries are generated with ``pfr export`` (no device connection needed) and written
  directly to their CMPA addresses over SWD using SEGGER J-Link Commander's ``LoadFile``
  command.
- The lifecycle state is inspected with ``nxpdebugmbox cmd get-crp`` (Debug Mailbox
  ``GET_CRP_LEVEL``), not with ``blhost get-property``.
- The mass-erase itself is performed with ``nxpdebugmbox cmd erase`` over the Debug Mailbox.

From SPSDK's perspective the process includes:

- Erasing the device while it is still in the open (DEVELOP) lifecycle state, using the plain
  (unconditional) Debug Mailbox ``erase`` command.
- Generating a CMPA_PSWD binary containing the ``PASSWORD_HEADER`` marker and the
  ``ERASE_PASSWORD`` value, and a CMPA_LC binary requesting the IN_FIELD1 lifecycle, with
  ``pfr export``.
- Writing both binaries to their CMPA addresses over SWD with J-Link Commander.
- Confirming the new lifecycle with ``nxpdebugmbox cmd get-crp``.
- Issuing the password-protected mass-erase with ``nxpdebugmbox cmd erase --password``.

## WARNING!

This configuration is used only for demonstration purposes. For a final security configuration go
through all configuration possibilities and define your own specific password. Once the password
is programmed and the device is moved to IN_FIELD1, the only way to erase the flash is through
this exact password, so make sure to keep a copy of it.

## 1. Prerequisites

- SPSDK is needed with examples extension. ``pip install spsdk[examples]`` (Please refer to the
  [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- An MCXC353 board.
- A J-Link probe connected to the SWD/JTAG header of the board (used for both the CMPA
  provisioning and the Debug Mailbox steps below).

### 1.1 Let's prepare the environment

In [ ]:
# Import required modules
import os

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

# Setup workspace
WORKSPACE = "workspace/"
DATA_DIR = "inputs/"
VERBOSITY = "-v"

# Create output directory
os.makedirs(WORKSPACE, exist_ok=True)

# Device configuration
FAMILY = "mcxc353"
INTERFACE = "jlink"
# Erase password as a 32-character hex string. The same string is later passed to
# `nxpdebugmbox cmd erase --password`.
ERASE_PASSWORD = "0xAABBCC00AABBCC00AABBCC00AABBCC00"
CMPA_ERASE_PSWD_CONFIG = os.path.join(DATA_DIR, "mcxc353_cmpa_erase_pswd.yaml")
CMPA_LC_CONFIG = os.path.join(DATA_DIR, "mcxc353_cmpa_lc.yaml")
CMPA_PSWD_BIN = os.path.join(WORKSPACE, "cmpa_pswd.bin")
CMPA_LC_BIN = os.path.join(WORKSPACE, "cmpa_lc.bin")

# Path to the JLink.exe executable used for programming the device.
# If JLink.exe is available on the system PATH, "JLink.exe" is sufficient.
# Otherwise, replace this with the full path to your J-Link installation, e.g.:
JLINK_EXE = "JLink.exe"

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


## Step 1: Check the current lifecycle

Open the Debug Mailbox and read the lifecycle state (``get-crp``). On a fresh device this should
be ``DEVELOP``, which allows the unconditional mass-erase used in the next step.

In [2]:
%! nxpdebugmbox -i $INTERFACE $VERBOSITY cmd -f $FAMILY start
%! nxpdebugmbox -i $INTERFACE $VERBOSITY -n cmd -f $FAMILY get-crp

nxpdebugmbox -i jlink -v cmd -f mcxc353 start 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
Start Debug Mailbox succeeded
nxpdebugmbox -i jlink -v -n cmd -f mcxc353 get-crp 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
CRP level is: DEVELOP.
Get CRP Level succeeded


## Step 2: Erase flash in DEVELOP lifecycle

Erase the entire flash while the device is still in the open lifecycle state, using the plain
(unconditional) Debug Mailbox ``erase`` command - no password is required or accepted in
``DEVELOP``. After the transition to IN_FIELD1 in Step 5, this unconditional erase will no
longer be available - only the password-protected mailbox erase from Step 8 will be.

In [3]:
%! nxpdebugmbox -i $INTERFACE $VERBOSITY cmd -f $FAMILY erase

nxpdebugmbox -i jlink -v cmd -f mcxc353 erase 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
INFO:spsdk.dat.debug_mailbox: Applying non-standard delay: 0.5 seconds to execute ERASE_FLASH
Mass flash erase succeeded


## Step 3: Generate the CMPA_PSWD and CMPA_LC binaries

Generate the CMPA_PSWD binary from [`inputs/mcxc353_cmpa_erase_pswd.yaml`](./inputs/mcxc353_cmpa_erase_pswd.yaml).
It contains the ``PASSWORD_HEADER`` marker (``0x5057``) and the 128-bit ``ERASE_PASSWORD`` token.
The marker is mandatory - the ROM refuses every password attempt with status ``0x00100003``
("Command failed") if the marker word is not present in CMPA.

Also generate the CMPA_LC binary from [`inputs/mcxc353_cmpa_lc.yaml`](./inputs/mcxc353_cmpa_lc.yaml),
requesting the IN_FIELD1 lifecycle state. Both binaries are produced by ``pfr export``, which does
not require a device connection - only the subsequent ``LoadFile`` step in Step 4 talks to the
board.

In [4]:
%! pfr $VERBOSITY export -c $CMPA_ERASE_PSWD_CONFIG -o $CMPA_PSWD_BIN
%! pfr $VERBOSITY export -c $CMPA_LC_CONFIG -o $CMPA_LC_BIN

assert os.path.exists(CMPA_PSWD_BIN)
assert os.path.exists(CMPA_LC_BIN)

pfr -v export -c inputs/mcxc353_cmpa_erase_pswd.yaml -o workspace/cmpa_pswd.bin 
INFO:spsdk.pfr.pfr:
+==0x0000_0000= mcxc353 ================+
|              Size: 68 B               |
|             Pattern: 0x0              |
|+==0x0000_0000= PASSWORD_HEADER ======+|
||              Size: 4 B              ||
||           Password header           ||
|+==0x0000_0003========================+|
|+==0x0000_0004= Reserved 0x00004 =====+|
||              Size: 4 B              ||
|| This field is reserved for internal ||
||                 use                 ||
|+==0x0000_0007========================+|
|+==0x0000_0008= Reserved 0x00008 =====+|
||              Size: 4 B              ||
|| This field is reserved for internal ||
||                 use                 ||
|+==0x0000_000b========================+|
|+==0x0000_000c= Reserved 0x0000C =====+|
||              Size: 4 B              ||
|| This field is reserved for internal ||
||                 use                 ||
|+==0x0000_000f==

## Step 4: Program CMPA over SWD with J-Link Commander

Because MCXC353 has no ISP mode, both binaries must be written directly to their CMPA addresses
over the debug interface. The commands below have been captured into a J-Link Commander script
([`inputs/program_cmpa.jlink`](./inputs/program_cmpa.jlink)) that connects to the target, halts the
core, writes the CMPA_PSWD binary to the password page (`0x0100_2800`) and the CMPA_LC binary to
the lifecycle page (`0x0100_3000`), then resets:

```
LoadFile workspace/cmpa_pswd.bin 0x01002800
LoadFile workspace/cmpa_lc.bin 0x01003000
```

**This lifecycle transition to IN_FIELD1 is irreversible.** After the reset triggered by this
script, the ROM will only accept the password-protected mailbox erase (Step 8), not the plain
one used in Step 2.

Before running the cell below, make sure `JLink.exe` can be found - either by having it on the
system `PATH`, or by setting the full path in the `JLINK_EXE` variable in Step 1.1. To check if it
is on the `PATH`:

- PowerShell: `Get-Command JLink.exe`
- cmd.exe: `where JLink.exe`

If neither finds it, look for the installation under
`C:\Program Files\SEGGER\JLink_<version>\JLink.exe` (Windows) and set `JLINK_EXE` accordingly, or
add that folder to your `PATH`.

In [5]:
JLINK_EXE = r"C:\Program Files\SEGGER\JLink_V798h\JLink.exe"
%! "$JLINK_EXE" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/program_cmpa.jlink"

"C:\Program Files\SEGGER\JLink_V798h\JLink.exe" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/program_cmpa.jlink" 
SEGGER J-Link Commander V7.98h (Compiled Sep 11 2024 14:32:20)
DLL version V7.98h, compiled Sep 11 2024 14:31:28

J-Link Commander will now exit on Error

J-Link Command File read successfully.
Processing script file...
J-Link>device MCXC353
J-Link connection not established yet but required for command.
Connecting to J-Link via USB...O.K.
Firmware: J-Link V11 compiled Sep  3 2024 10:40:48
Hardware version: V11.00
J-Link uptime (since boot): 0d 00h 08m 56s
S/N: 851006367
License(s): RDI, FlashBP, FlashDL, JFlash, GDB
USB speed mode: High speed (480 MBit/s)
VTref=3.313V
J-Link>si SWD
Selecting SWD as current target interface.
J-Link>speed 4000
Selecting 4000 kHz as target interface speed
J-Link>connect
Device "MCXC353" selected.


Connecting to target via SWD
InitTarget() start
******************************************************
J-Link script: MCX Cortex-M23 core J-Link s

## Step 5: Verify the new lifecycle

Re-open the Debug Mailbox and read the lifecycle again with ``get-crp`` to confirm the device is
now in IN_FIELD1.

In [6]:
%! nxpdebugmbox -i $INTERFACE $VERBOSITY cmd -f $FAMILY start
%! nxpdebugmbox -i $INTERFACE $VERBOSITY -n cmd -f $FAMILY get-crp

nxpdebugmbox -i jlink -v cmd -f mcxc353 start 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
Start Debug Mailbox succeeded
nxpdebugmbox -i jlink -v -n cmd -f mcxc353 get-crp 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
CRP level is: IN_FIELD1.
Get CRP Level succeeded


## Step 6: Mass-erase flash with password

Issue the password-protected ``ERASE_FLASH`` mailbox command. The host sends the request word
``0x00040003`` (``paramlen=4`` | ``cmd=ERASE_FLASH``) followed by the four 32-bit password
words, acknowledging each ``0xNNNN_A5A5`` response from the device. On success the device
returns status ``0`` and reboots after the erase completes.

The hex string passed to ``--password`` must exactly match the ``ERASE_PASSWORD`` value
programmed in Step 3 - SPSDK packs the password as four big-endian 32-bit words so the
hex string maps 1:1 onto the wire words.

**Expected result:** ``Mass flash erase succeeded``. If you see
``SPSDKAppError: Mass flash erase failed: ... Command failed`` (status ``0x00100003``), either
the ``PASSWORD_HEADER`` marker is missing in CMPA (re-check Step 3/4) or the password bytes do
not match (re-check the value below against Step 3).

In [7]:
%! nxpdebugmbox -i $INTERFACE $VERBOSITY cmd -f $FAMILY erase --password $ERASE_PASSWORD

nxpdebugmbox -i jlink -v cmd -f mcxc353 erase --password 0xAABBCC00AABBCC00AABBCC00AABBCC00 
  #   Interface   Id          Description                  Architecture   Protocols  
-------------------------------------------------------------------------------------
  0   Jlink       851006367   Segger J-Link Compact PLUS   arm-cortex     swd, jtag  
INFO:spsdk.dat.debug_mailbox: Applying non-standard delay: 0.5 seconds to execute ERASE_FLASH
Mass flash erase succeeded


## Summary

This notebook demonstrated the full password-protected mass-erase flow for the MCXC353 device.
Because MCXC353 does not support ISP mode, every step below uses either the Debug Mailbox
(``nxpdebugmbox``) or direct SWD programming (J-Link Commander) instead of ``blhost``/``pfr
write``/``pfr read``:

1. Verified the open lifecycle (``DEVELOP``) with ``nxpdebugmbox cmd get-crp``.
2. Erased flash unconditionally via the Debug Mailbox while the device was still unrestricted.
3. Generated the CMPA_PSWD binary (``PASSWORD_HEADER`` marker + ``ERASE_PASSWORD`` token) and the
   CMPA_LC binary (IN_FIELD1) with ``pfr export``.
4. Programmed both binaries directly to CMPA over SWD using J-Link Commander.
5. Verified the new lifecycle (``IN_FIELD1``) with ``nxpdebugmbox cmd get-crp``.
6. Performed the password-protected mass-erase via ``nxpdebugmbox cmd erase --password``.

The device is now erased and remains in ``IN_FIELD1`` with the same password protection in
place for the next provisioning cycle.